# fedOPD — inspect the fetched RTT files

Run after `fetch_dataset.ipynb`. Looks at what actually landed on disk:

1. file inventory — every month, its size, its naming style
2. size variation — outlier months against a rolling median
3. filename variations — NHS renamed these files repeatedly over 8 years
4. header/column variations — do all 96 months share one schema?
5. row counts per month
6. **delete the zips**, keeping only the extracted csvs (last cell, run deliberately)

In [ ]:
import re                                    # regex — parses month + naming template out of filenames
from pathlib import Path                     # filesystem paths
from datetime import datetime                # build a real date from 'OCTOBER' + '2019'

import pandas as pd                          # the inventory table
import matplotlib.pyplot as plt              # size / row-count charts

DATA = Path("../data")                       # same layout as fetch_dataset.ipynb
RTT_ZIPS = DATA / "rtt_zips"                 # the downloaded zips (deleted in section 6)
RTT_CSV = DATA / "rtt_csv"                   # the extracted csvs — what we analyse here

csvs = sorted(RTT_CSV.glob("*.csv"))         # all 96 extracted files
zips = sorted(RTT_ZIPS.glob("*.zip"))        # all 96 archives they came from
print(len(csvs), "csvs,", len(zips), "zips")

## 1. File inventory

Pull the month out of each filename so files sort chronologically rather than alphabetically. Two naming styles are already visible: some files start with a `YYYYMMDD-` stamp (`20191031-RTT-OCTOBER-2019-full-extract.csv`), others start straight in with `RTT-MONTH-YYYY`.

In [ ]:
MONTHS = "JANUARY FEBRUARY MARCH APRIL MAY JUNE JULY AUGUST SEPTEMBER OCTOBER NOVEMBER DECEMBER".split()

rows = []                                              # one dict per file
for p in csvs:
    n = p.name.upper()                                 # uppercase so the month match is case-proof
    # every filename carries 'MONTHNAME-YYYY' somewhere, e.g. 'RTT-OCTOBER-2019-full-extract'
    m = re.search(r"(" + "|".join(MONTHS) + r")-(\d{4})", n)   # group1 = month word, group2 = year
    rows.append({
        "file": p.name,
        # MONTHS.index(...) + 1 turns 'OCTOBER' into 10, giving a sortable first-of-month date
        "month": datetime(int(m.group(2)), MONTHS.index(m.group(1)) + 1, 1),
        "mb": p.stat().st_size / 1e6,                  # size on disk in MB
        "has_date_prefix": bool(re.match(r"^\d{8}-", p.name)),   # leading '20191031-'?
        "revised": "revised" in p.name.lower(),        # NHS reissued that month
    })

files = pd.DataFrame(rows).sort_values("month").reset_index(drop=True)   # chronological order
print(len(files), "files |", files.month.min().date(), "->", files.month.max().date())
print(f"total {files.mb.sum() / 1000:.1f} GB")         # sum of MB / 1000 = GB
files.head()

In [ ]:
# is any month missing or duplicated across the 96-month window?
expected = pd.date_range("2018-04-01", "2026-03-01", freq="MS")   # MS = month start, 96 entries
print("expected:", len(expected), "| got:", files.month.nunique())
# .difference() = months we expected but don't have
print("missing:", [d.strftime('%b%Y') for d in expected.difference(files.month)])
# any month appearing twice would mean two csvs for the same period (e.g. original + revised both extracted)
print("duplicated months:", files.month.value_counts().loc[lambda s: s > 1].to_dict())

## 2. Size variation

Sizes should drift upward as more providers and specialties report, with a visible dip through the covid months when reporting collapsed.

In [ ]:
print(files.mb.describe().round(1).to_string())        # count/mean/std/min/quartiles/max
print("\nsmallest 5:")
print(files.nsmallest(5, "mb")[["month", "file", "mb"]].to_string(index=False))
print("\nlargest 5:")
print(files.nlargest(5, "mb")[["month", "file", "mb"]].to_string(index=False))

In [ ]:
# flag months far from their neighbours — a centred 12-month rolling median is a fair local baseline
files["rolling_med"] = files.mb.rolling(12, center=True, min_periods=3).median()
files["ratio"] = files.mb / files.rolling_med          # 1.0 = exactly typical for its period
odd = files[(files.ratio < 0.75) | (files.ratio > 1.33)]   # >25% either side
print("months >25% away from their 12-month rolling median:")
print(odd[["month", "file", "mb", "rolling_med", "ratio"]].round(2).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(files.month, files.mb, marker="o", ms=3, lw=1, label="file size")          # actual per month
ax.plot(files.month, files.rolling_med, lw=1.5, ls="--", label="12-month rolling median")   # baseline
ax.set_ylabel("MB")
ax.set_title("RTT full-extract csv size by month")
ax.legend()
plt.tight_layout()

## 3. Filename variations

Strip the month, year and date-stamp out of each name to see the distinct templates NHS used. This is what any later globbing code has to survive — the zip names alone included oddities like `Full-CSV-data-file-Sep18-revised-ZIP-K.zip` (empty size token) and random suffixes such as `-SrRW6y`.

In [ ]:
def template(name):
    s = re.sub(r"^\d{8}-", "<DATE>-", name)                        # leading yyyymmdd stamp -> <DATE>
    s = re.sub(r"(" + "|".join(MONTHS) + r")", "<MONTH>", s.upper())   # OCTOBER -> <MONTH>
    s = re.sub(r"\d{4}", "<YEAR>", s)                              # 2019 -> <YEAR>
    return s                                                       # what's left is the naming template

files["template"] = files.file.map(template)          # apply to every filename
print(files.template.value_counts().to_string())      # how many months use each template
print("\nwith date prefix:", int(files.has_date_prefix.sum()), "| without:", int((~files.has_date_prefix).sum()))
print("marked 'revised':", int(files.revised.sum()))

In [ ]:
# when was each naming template in use? the min/max dates show the changeover points
print(files.groupby("template").month.agg(["min", "max", "count"]).to_string())

## 4. Header / schema variations

The one that matters: if columns differ month to month, the phase-2 (demand) and phase-4 (flow) extractions have to cope. Reading just the first line of each file is cheap — no need to parse 7.7 GB.

In [ ]:
headers = {}                                          # filename -> tuple of column names
for _, r in files.iterrows():
    with open(RTT_CSV / r.file, newline="") as f:
        line = f.readline().rstrip("\r\n")            # header row only, drop the line ending
    # naive split on ',' is fine here: the header itself has no commas inside its quoted names
    headers[r.file] = tuple(c.strip().strip('"') for c in line.split(","))   # strip quotes + spaces

files["n_cols"] = files.file.map(lambda f: len(headers[f]))    # how many columns that month has
files["schema"] = files.file.map(lambda f: hash(headers[f]))   # hash the tuple = schema fingerprint

print("distinct column counts:", sorted(files.n_cols.unique()))
print("distinct schemas:", files.schema.nunique())    # 1 would mean perfectly stable over 8 years
print()
# group the months by fingerprint: how many months, and the date range each schema covers
print(files.groupby("schema").agg(months=("month", "count"),
                                  first=("month", "min"),
                                  last=("month", "max"),
                                  n_cols=("n_cols", "first")).to_string())

In [ ]:
# what actually differs: compare every other schema against the most common one
base_schema = files.schema.value_counts().index[0]    # the fingerprint used by the most months
base_file = files[files.schema == base_schema].file.iloc[0]   # any file having it
base = headers[base_file]                             # its column tuple = the baseline
print("baseline:", base_file, f"({len(base)} cols)\n")

for s in files.schema.unique():
    if s == base_schema:
        continue                                      # skip the baseline itself
    f = files[files.schema == s].file.iloc[0]         # one representative file per variant
    cols = headers[f]
    print(f, f"({len(cols)} cols)")
    print("  only here :", [c for c in cols if c not in base])   # columns this variant adds
    print("  missing   :", [c for c in base if c not in cols])   # columns it drops
    print()

In [ ]:
# the columns fedOPD actually needs downstream — confirm they exist in every single month
needed = ["Period", "Provider Parent Org Code", "Provider Org Code",
          "RTT Part Type", "RTT Part Description", "Treatment Function Code"]
for c in needed:
    missing = [f for f, cols in headers.items() if c not in cols]   # files lacking this column
    print(f"{c:32s} present in {96 - len(missing):2d}/96", "" if not missing else f"| MISSING: {missing[:3]}")

## 5. Row counts

Slow cell — it streams all ~7.7 GB to count newlines. A few minutes.

In [ ]:
def count_lines(path):
    n = 0
    with open(path, "rb") as f:                       # binary: we only count b'\n' bytes
        while chunk := f.read(1 << 24):               # walrus: read 16 MB, stop when b'' (EOF)
            n += chunk.count(b"\n")                   # newlines in this chunk
    return n

files["rows"] = [count_lines(RTT_CSV / f) - 1 for f in files.file]   # minus 1 for the header row
print(f"total rows: {files.rows.sum():,}")            # ',' = thousands separator
print(files.rows.describe().round(0).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(files.month, files.rows / 1000, marker="o", ms=3, lw=1)   # thousands of rows per month
ax.set_ylabel("rows (thousands)")
ax.set_title("RTT rows per month")
plt.tight_layout()

files.to_csv(DATA / "reference" / "rtt_file_inventory.csv", index=False)   # save the whole inventory
print("inventory written to data/reference/rtt_file_inventory.csv")

## 6. Delete the zips

The csvs are the only copy needed from here on, so the ~330 MB of archives is a duplicate of data we already have extracted.

One-way step — run it only once sections 1-4 look right, because re-downloading means a full rerun of `fetch_dataset.ipynb`.

In [ ]:
freed = sum(z.stat().st_size for z in zips)       # total bytes about to be reclaimed
for z in zips:
    z.unlink()                                    # delete each zip file
RTT_ZIPS.rmdir()                                  # remove the now-empty folder
print(f"deleted {len(zips)} zips, freed {freed / 1e6:.0f} MB")
print("csvs still on disk:", len(list(RTT_CSV.glob('*.csv'))))   # should still be 96